# Time Series Benchmarking Framework

Interactive notebook for running forecast horse-races.  
**Workflow:** Setup series → Configure knobs → Run benchmark → Inspect results

In [1]:
import os
import importlib.util
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from benchmark import (
    TimeSeries, SeriesRegistry,
    MeanForecaster, ARIMAForecaster, BayesianARForecaster, SSAForecaster, TimesFMForecaster,
    BenchmarkRunner, BenchmarkResults,
    ReplicatedBenchmarkRunner, ReplicatedBenchmarkResults,
)
spec = importlib.util.spec_from_file_location("local_secrets", "./secrets.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
HF_token = mod.HF_token
os.environ["HF_TOKEN"] = HF_token

## 1. Setup Phase — Register Series

Load the default synthetic DGPs and (optionally) register your own.

In [2]:
SeriesRegistry.clear()
SeriesRegistry.register_defaults(n=100, seed=42)

print("Available series:", SeriesRegistry.list_available())

Available series: ['AR(2)', 'AR(5)', 'ARMA(2,2)', 'ARMA(5,5)', 'Seasonal(12)']


In [3]:
# --- Register your own series (uncomment / adapt) ---

# From a CSV:# SeriesRegistry.register_from_csv("SP500", path="data/sp500.csv", column="Close")

# From a raw array:
# my_values = np.random.randn(500).cumsum()
# SeriesRegistry.register("Random Walk", TimeSeries.from_array(my_values, name="Random Walk"))

print("Available series:", SeriesRegistry.list_available())

Available series: ['AR(2)', 'AR(5)', 'ARMA(2,2)', 'ARMA(5,5)', 'Seasonal(12)']


## 2. Monte Carlo Replications — Same DGP, Multiple Seeds

Run the *same* DGP with many different random seeds so each replication
draws new AR/MA coefficients **and** new innovations.  This lets you
compare forecasters on the *distribution* of performance rather than a
single lucky/unlucky draw.

In [6]:
# ── DGP and replication settings ────────────────────────────────

dgp_options = {
    "AR(2)":       lambda seed: TimeSeries.from_ar(p=2, n=100, seed=seed),
    "AR(5)":       lambda seed: TimeSeries.from_ar(p=5, n=100, seed=seed),
    "ARMA(2,2)":   lambda seed: TimeSeries.from_arma(p=2, q=2, n=100, seed=seed),
    "ARMA(5,5)":   lambda seed: TimeSeries.from_arma(p=5, q=5, n=100, seed=seed),
    "Seasonal(12)": lambda seed: TimeSeries.from_seasonal(n=100, period=12, seed=seed),
}

mc_dgp_dropdown = widgets.Dropdown(
    options=list(dgp_options.keys()),
    description='DGP:',
    style={'description_width': 'initial'},
)

mc_n_replications = widgets.IntSlider(
    value=10, min=2, max=50, step=1,
    description='Replications:',
    style={'description_width': 'initial'},
)

mc_base_seed = widgets.IntSlider(
    value=0, min=0, max=1000, step=1,
    description='Base seed:',
    style={'description_width': 'initial'},
)

mc_horizon = widgets.IntSlider(
    value=1, min=1, max=30, step=1,
    description='Horizon (h):',
    style={'description_width': 'initial'},
)

mc_k_first = widgets.IntSlider(
    value=360, min=10, max=500, step=10,
    description='Training window (k_first):',
    style={'description_width': 'initial'},
)

# Forecaster toggles (parallel controls to single-run section)
mc_use_mean = widgets.Checkbox(value=True, description='Mean (baseline)')
mc_use_arima = widgets.Checkbox(value=True, description='ARIMA')
mc_use_bayes_ar = widgets.Checkbox(value=True, description='Bayes AR (ridge / MN)')
mc_bayes_ar_p = widgets.IntSlider(value=2, min=1, max=15, description='  BayesAR p:')
mc_bayes_ar_lambda = widgets.FloatLogSlider(
    value=1.0, base=10, min=-3, max=3, step=0.25,
    description='  BayesAR λ:',
    style={'description_width': 'initial'},
)
mc_bayes_ar_mode = widgets.Dropdown(
    options=[('Ridge (λI)', 'ridge'), ('Minnesota', 'minnesota')],
    value='ridge',
    description='  BayesAR mode:',
    style={'description_width': 'initial'},
)
mc_bayes_ar_mn_decay = widgets.FloatSlider(
    value=2.0, min=0.0, max=4.0, step=0.25,
    description='  MN lag exp:',
    style={'description_width': 'initial'},
)
mc_bayes_ar_mn_rw = widgets.Checkbox(value=True, description='  MN RW mean on φ₁')
mc_use_ssa = widgets.Checkbox(value=True, description='SSA')
mc_use_timesfm = widgets.Checkbox(value=True, description='TimesFM 2.5')

display(
    widgets.HTML('<h4>Monte Carlo settings</h4>'),
    mc_dgp_dropdown, mc_n_replications, mc_base_seed,
    mc_horizon, mc_k_first,
    widgets.HTML('<h4>Forecasters</h4>'),
    mc_use_mean, mc_use_arima,
    mc_use_bayes_ar, mc_bayes_ar_p, mc_bayes_ar_lambda,
    mc_bayes_ar_mode, mc_bayes_ar_mn_decay, mc_bayes_ar_mn_rw,
    mc_use_ssa, mc_use_timesfm,
)

HTML(value='<h4>Monte Carlo settings</h4>')

Dropdown(description='DGP:', options=('AR(2)', 'AR(5)', 'ARMA(2,2)', 'ARMA(5,5)', 'Seasonal(12)'), style=Descr…

IntSlider(value=10, description='Replications:', max=50, min=2, style=SliderStyle(description_width='initial')…

IntSlider(value=0, description='Base seed:', max=1000, style=SliderStyle(description_width='initial'))

IntSlider(value=1, description='Horizon (h):', max=30, min=1, style=SliderStyle(description_width='initial'))

...

Dropdown(description='  BayesAR mode:', options=(('Ridge (λI)', 'ridge'), ('Minnesota', 'minnesota')), style=D…

FloatSlider(value=2.0, description='  MN lag exp:', max=4.0, step=0.25, style=SliderStyle(description_width='i…

Checkbox(value=True, description='  MN RW mean on φ₁')

Checkbox(value=True, description='SSA')

Checkbox(value=True, description='TimesFM 2.5')

In [7]:
# ── Run Monte Carlo replications ────────────────────────────────

mc_output = widgets.Output()
mc_button = widgets.Button(description='Run Replications', button_style='success')

mc_results: ReplicatedBenchmarkResults | None = None

def _on_mc_run(btn):
    global mc_results
    with mc_output:
        clear_output(wait=True)

        forecasters = []
        if mc_use_mean.value:
            forecasters.append(MeanForecaster(window=50))
        if mc_use_arima.value:
            forecasters.append(ARIMAForecaster(order=(2, 0, 0)))
        if mc_use_bayes_ar.value:
            forecasters.append(
                BayesianARForecaster(
                    p=mc_bayes_ar_p.value,
                    prior_precision=float(mc_bayes_ar_lambda.value),
                    prior_mode=str(mc_bayes_ar_mode.value),
                    minnesota_lag_decay_exponent=float(mc_bayes_ar_mn_decay.value),
                    minnesota_center_rw=mc_bayes_ar_mn_rw.value,
                )
            )
        if mc_use_ssa.value:
            forecasters.append(SSAForecaster())
        if mc_use_timesfm.value:
            forecasters.append(TimesFMForecaster(max_context=512))

        if not forecasters:
            print('Select at least one forecaster.')
            return

        dgp_factory = dgp_options[mc_dgp_dropdown.value]

        runner = ReplicatedBenchmarkRunner(
            dgp_factory=dgp_factory,
            forecasters=forecasters,
            n_replications=mc_n_replications.value,
            base_seed=mc_base_seed.value,
            k_first=mc_k_first.value,
            horizon=mc_horizon.value,
            verbose=True,
        )

        print(f'Running {mc_n_replications.value} replications of '
              f'"{mc_dgp_dropdown.value}" '
              f'(k_first={mc_k_first.value}, h={mc_horizon.value})...\n')
        mc_results = runner.run()

        print('\n── Aggregate Metrics (mean ± std across seeds) ──')
        display(mc_results.aggregate_metrics())
        print('\n── Pooled Diebold-Mariano (SE loss) ──')
        display(mc_results.pooled_diebold_mariano())

mc_button.on_click(_on_mc_run)
display(mc_button, mc_output)

Button(button_style='success', description='Run Replications', style=ButtonStyle())

Output()

### Monte Carlo Results — Plots

### Monte Carlo — Coverage & Phase 2

Mean empirical coverage across seeds ± one std (when multiple seeds); ``N_valid_mean`` averages evaluation-point counts per seed. **Phase 2** adds ``mean_log_score_kde_mean`` / ``_std`` and sharpness–calibration columns aggregated across seeds.


In [6]:
if mc_results is not None:
    display(mc_results.coverage_table())
    if any(r.quantile_predictions for r in mc_results.per_seed):
        print("\n── Phase 2 (aggregated across seeds) ──")
        display(mc_results.probabilistic_summary_phase2(nominal=0.9))
    else:
        print("No quantile forecasts in any replication (enable TimesFM or another quantile model).")
else:
    print("Run the Monte Carlo replications first (cell above).")


Run the Monte Carlo replications first (cell above).


In [7]:
if mc_results is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    mc_results.plot_metric_distribution("mse", ax=axes[0])
    mc_results.plot_pooled_cumulative_error(ax=axes[1])
    plt.tight_layout()
    plt.show()

    # Wide scorecard: per-seed MSE/MAE, pooled metrics, ratios vs best (pooled)
    print("\n── Monte Carlo scorecard (MSE & MAE per seed, pooled, rel vs best) ──")
    _mc_sc = mc_results.replication_scorecard()
    _num = [c for c in _mc_sc.columns if c != "Forecaster"]
    try:
        display(
            _mc_sc.style.format({c: "{:,.5f}" for c in _num}, na_rep="—")
            .set_properties(subset=_num, **{"text-align": "right"})
            .set_table_styles(
                [{"selector": "th", "props": [("text-align", "left")]}],
                overwrite=False,
            )
        )
    except Exception:
        display(_mc_sc.round(5))
else:
    print('Run the Monte Carlo replications first (cell above).')

Run the Monte Carlo replications first (cell above).
